# Logistics Data Analysis – Week 1

Strategic planning and data exploration for an e-commerce logistics scenario.

**KPIs:** On-Time Delivery Rate, Average Delivery Lead Time, Freight Cost per Order.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)


In [ ]:
# Load the public Olist dataset after downloading it into ../data/
orders = pd.read_csv('../data/olist_orders_dataset.csv')
items = pd.read_csv('../data/olist_order_items_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')

print('Orders:', orders.shape)
print('Items:', items.shape)
print('Customers:', customers.shape)


In [ ]:
# Inspect missing values
missing = orders.isna().sum().sort_values(ascending=False)
display(missing.head(10))


In [ ]:
# Convert timestamps
date_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

delivered = orders.dropna(
    subset=['order_delivered_customer_date', 'order_estimated_delivery_date']
).copy()

delivered['delivery_days'] = (
    delivered['order_delivered_customer_date']
    - delivered['order_purchase_timestamp']
).dt.total_seconds() / 86400

delivered['delay_days'] = (
    delivered['order_delivered_customer_date']
    - delivered['order_estimated_delivery_date']
).dt.total_seconds() / 86400

delivered['is_late'] = delivered['delay_days'] > 0
delivered[['delivery_days', 'delay_days', 'is_late']].describe()


In [ ]:
# KPI calculation
on_time_rate = (~delivered['is_late']).mean() * 100
avg_delivery_days = delivered['delivery_days'].mean()
freight_by_order = items.groupby('order_id')['freight_value'].sum()
avg_freight = freight_by_order.mean()

print(f'On-time delivery rate: {on_time_rate:.2f}%')
print(f'Average delivery lead time: {avg_delivery_days:.2f} days')
print(f'Average freight cost per order: {avg_freight:.2f}')


In [ ]:
# Delivery-time distribution
plt.figure(figsize=(9, 5))
sns.histplot(delivered['delivery_days'].dropna(), bins=40, kde=True)
plt.title('Distribution of Delivery Lead Time')
plt.xlabel('Delivery time (days)')
plt.ylabel('Number of orders')
plt.tight_layout()
plt.show()


## Next analytical stages

- Join customer/seller/geolocation information.
- Analyze KPIs by region and month.
- Build a regression model for delivery lead time.
- Apply clustering to regional logistics behavior.
- Prototype vehicle routing with OR-Tools.
